# Assignment 2 – Multiple Linear Regression

## Overview

This notebook demonstrates **multiple linear regression** with two independent features predicting a continuous output.

### Problem Statement
- **True Relationship:** $y = 4.0x_1 + 2.5x_2 + 10$
- **Data:** 150 samples with:
    - $x_1 \in [0, 50]$ (feature 1)
    - $x_2 \in [0, 30]$ (feature 2)
    - Added Gaussian noise ($\sigma = 15$)
- **Goal:** Learn coefficients $a_1=4.0$, $a_2=2.5$, and intercept $c=10$ using multiple linear regression

### Method
1. Generate synthetic data from the true equation plus noise
2. Split data into train (80%) and test (20%) sets
3. Fit Linear Regression with two features
4. Compare learned coefficients to true values
5. Evaluate using MAE, MSE, RMSE, R²
6. Assess generalization (overfitting/underfitting)
7. Visualize predictions, residuals, and feature importance

### Key Difference from Simple Regression
- **Simple (univariate):** One input feature → one output (y = ax + b)
- **Multiple (multivariate):** Two or more input features → one output (y = a₁x₁ + a₂x₂ + ... + c)
- Each feature has its own coefficient learned from data

## Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("     All libraries imported successfully")

## Step 1: Define True Equation Parameters

We formulate a linear equation with two features and known coefficients.

**The Equation:**
$$y = a_1 x_1 + a_2 x_2 + c$$

where:
- $a_1 = 4.0$ is the weight of feature $x_1$
- $a_2 = 2.5$ is the weight of feature $x_2$
- $c = 10$ is the intercept (bias term)

In [ ]:
# Define true coefficients
a1_true = 4.0     # Coefficient for x1
a2_true = 2.5     # Coefficient for x2
c_true    = 10.0     # Intercept

print(f"\n{'='*50}")
print(f"True Equation Parameters")
print(f"{'='*50}")
print(f"y = {a1_true}·x₁ + {a2_true}·x₂ + {c_true}")
print(f"{'='*50}\n")

## Step 2: Generate Feature Values

Create two independent feature variables with random values within specified ranges:
- $x_1 \sim \text{Uniform}(0, 50)$ — Feature 1 range
- $x_2 \sim \text{Uniform}(0, 30)$ — Feature 2 range

**Why uniform distribution?** Ensures even coverage across the feature space.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of samples
n = 150

# Generate features from uniform distributions
x1 = np.random.uniform(0, 50, n)     # Feature 1: uniformly distributed in [0, 50]
x2 = np.random.uniform(0, 30, n)     # Feature 2: uniformly distributed in [0, 30]

print(f"Generated {n} samples")
print(f"\nFeature x₁ statistics:")
print(f"    Range: [{x1.min():.2f}, {x1.max():.2f}]")
print(f"    Mean:    {x1.mean():.2f}")
print(f"\nFeature x₂ statistics:")
print(f"    Range: [{x2.min():.2f}, {x2.max():.2f}]")
print(f"    Mean:    {x2.mean():.2f}")

## Step 3: Generate Target Values with Noise

**Process:**
1. Compute y values using the true equation
2. Add Gaussian random noise: $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ where $\sigma = 15$

**Why add noise?** Real-world data contains measurement errors and unobserved factors. This simulates realistic conditions.

**The observed data:**
$$y_{\text{observed}} = y_{\text{true}} + \varepsilon$$

In [ ]:
# Generate noise (Gaussian, mean=0, std=15)
noise = np.random.normal(0, 15, n)

# Compute y from true equation + noise
Y = a1_true * x1 + a2_true * x2 + c_true + noise

print(f"Generated target Y values with noise")
print(f"\nTarget Y statistics:")
print(f"    Range: [{Y.min():.2f}, {Y.max():.2f}]")
print(f"    Mean:    {Y.mean():.2f}")
print(f"    Std:     {Y.std():.2f}")
print(f"\nNoise level (σ): 15")

## Step 4: Prepare Data and Train/Test Split

**Data Preparation:**
- Stack $x_1$ and $x_2$ into a 2D feature matrix $X$ of shape (150, 2)

**Train/Test Split (80/20):**
- **Training set (120 samples):** Used to fit model parameters
- **Test set (30 samples):** Used to evaluate generalization
- **random_state=42:** Ensures reproducible splits

In [ ]:
# Stack features into 2D matrix
X = np.column_stack([x1, x2])

print(f"Feature matrix X shape: {X.shape}")
print(f"    [Row 0]: x₁={X[0,0]:.2f}, x₂={X[0,1]:.2f}")
print(f"    [Row 1]: x₁={X[1,0]:.2f}, x₂={X[1,1]:.2f}")

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
     X, Y, test_size=0.20, random_state=42
)

print(f"\n{'─'*45}")
print(f"Train/Test Split (80/20)")
print(f"{'─'*45}")
print(f"Total samples         : {len(X)}")
print(f"Training samples     : {len(X_train)} ({100*len(X_train)/len(X):.0f}%)")
print(f"Test samples          : {len(X_test)} ({100*len(X_test)/len(X):.0f}%)")
print(f"\nTrain Y range: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Test Y range : [{y_test.min():.2f}, {y_test.max():.2f}]")

## Step 5: Model Training

**Algorithm:** Linear Regression (Ordinary Least Squares)

**What it learns:**
- Two coefficients: $w_1$ and $w_2$ (estimates of $a_1$ and $a_2$)
- One intercept: $w_0$ (estimate of $c$)
- These form the predicted model: $\hat{y} = w_1 x_1 + w_2 x_2 + w_0$

**How it works:**
Finds coefficients that minimize the sum of squared residuals:
$$\min \sum_{i=1}^{n} (y_i - (w_1 x_{1i} + w_2 x_{2i} + w_0))^2$$

In [ ]:
# Create and fit model
model = LinearRegression()
model.fit(X_train, y_train)

print("     Model training completed\n")
print(f"Learned Parameters:")
print(f"    Coefficient for x₁ (w₁): {model.coef_[0]:.4f}")
print(f"    Coefficient for x₂ (w₂): {model.coef_[1]:.4f}")
print(f"    Intercept (w₀)              : {model.intercept_:.4f}")
print(f"\nPredicted equation: y = {model.coef_[0]:.4f}·x₁ + {model.coef_[1]:.4f}·x₂ + {model.intercept_:.4f}")

## Step 6: Compare True vs. Learned Parameters

**Interpretation:**
- If learned values ≈ true values → model successfully recovered the underlying relationship
- Small differences are expected due to data noise
- Large differences might indicate data quality issues or model problems

In [ ]:
# Extract learned coefficients
a1_learned = model.coef_[0]
a2_learned = model.coef_[1]
c_learned = model.intercept_

print("\n" + "="*70)
print(f"{'Parameter':<20} {'True':>15} {'Learned':>15} {'Error':>15}")
print("="*70)

for name, true_val, learned_val in [
     ("a₁ (x₁ coeff)", a1_true, a1_learned),
     ("a₂ (x₂ coeff)", a2_true, a2_learned),
     ("c    (intercept)", c_true, c_learned),
]:
     error = abs(true_val - learned_val)
     pct_error = 100 * error / abs(true_val) if true_val != 0 else 0
     print(f"{name:<20} {true_val:>15.4f} {learned_val:>15.4f} {error:>15.4f} ({pct_error:.2f}%)")

print("="*70)
print(f"\n     Learned equation: y = {a1_learned:.4f}·x₁ + {a2_learned:.4f}·x₂ + {c_learned:.4f}")

## Step 7: Model Evaluation

**Metrics Used:**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Average absolute error (same units as y) |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Mean squared error (penalizes large errors) |
| **RMSE** | $\sqrt{MSE}$ | Root mean squared error (same units as y) |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Coefficient of determination (0→1, higher is better) |

**R² Interpretation:**
- R² = 1.0: Perfect fit (explains 100% of variance)
- R² = 0.9: Excellent (explains 90% of variance)
- R² = 0.5: Moderate (explains 50% of variance)
- R² = 0.0: Poor (explains 0% of variance)

In [ ]:
def evaluate(name, X_set, y_set):
     """
     Evaluate model on a given dataset.
     
     Parameters:
     -----------
     name : str
          Name of dataset (e.g., 'Train Set', 'Test Set')
     X_set : array, shape (n_samples, 2)
          Feature matrix [x1, x2]
     y_set : array, shape (n_samples,)
          Target values
     
     Returns:
     --------
     preds : array
          Model predictions
     """
     preds = model.predict(X_set)
     
     # Calculate metrics
     mae    = mean_absolute_error(y_set, preds)
     mse    = mean_squared_error(y_set, preds)
     rmse = np.sqrt(mse)
     r2     = r2_score(y_set, preds)
     
     # Display results
     print(f"{'─'*50}")
     print(f"{name:^50}")
     print(f"{'─'*50}")
     print(f"    MAE    (Mean Absolute Error)          : {mae:>10.4f}")
     print(f"    MSE    (Mean Squared Error)          : {mse:>10.4f}")
     print(f"    RMSE (Root Mean Squared Error)     : {rmse:>10.4f}")
     print(f"    R²     (Coefficient of Determ.)     : {r2:>10.4f}")
     print()
     
     return preds

print("\nMODEL PERFORMANCE EVALUATION\n")
train_preds = evaluate("Training Set", X_train, y_train)
test_preds    = evaluate("Test Set", X_test, y_test)

## Step 8: Diagnostic Analysis

### Overfitting vs. Underfitting

**Overfitting:** Model memorizes training noise instead of learning genuine patterns
- **Sign:** Train R² >> Test R², or test error much higher than train error
- **Cause:** Model too complex, too much noise in data
- **Fix:** Simplify model, add regularization, get more data

**Underfitting:** Model is too simple to capture the relationship
- **Sign:** Both train and test R² are low
- **Cause:** Model lacks capacity, features insufficient
- **Fix:** Add features, increase model complexity, train longer

### For This Model
- Train and test R² are **very close** → **No overfitting**     
- Both R² values are **high (>0.94)** → **No underfitting**     
- Error proportional to noise level (σ=15) → **Expected performance**     

In [ ]:
# Calculate metrics for diagnosis
train_r2 = r2_score(y_train, train_preds)
test_r2    = r2_score(y_test, test_preds)
train_mae = mean_absolute_error(y_train, train_preds)
test_mae    = mean_absolute_error(y_test, test_preds)

r2_gap = abs(train_r2 - test_r2)
mae_gap = abs(train_mae - test_mae)

print("\nDIAGNOSTIC SUMMARY\n")
print(f"R² Comparison:")
print(f"    Train R² : {train_r2:.4f}")
print(f"    Test R²    : {test_r2:.4f}")
print(f"    R² Gap     : {r2_gap:.4f}")
print(f"    Verdict    : {'     Small gap → Good generalization' if r2_gap < 0.05 else '✗ Large gap → Possible overfitting'}")

print(f"\nMAE Comparison:")
print(f"    Train MAE: {train_mae:.4f}")
print(f"    Test MAE : {test_mae:.4f}")
print(f"    MAE Gap    : {mae_gap:.4f}")
print(f"    Verdict    : {'     Similar errors → Good generalization' if mae_gap < 2.0 else '✗ Large gap → Possible overfitting'}")

print("\nOVERFITTING CHECK:")
if r2_gap < 0.05:
     print("         NO OVERFITTING - Train and test R² nearly identical")
else:
     print("    ✗ POSSIBLE OVERFITTING - Significant R² gap detected")

print("\nUNDERFITTING CHECK:")
if test_r2 > 0.85:
     print(f"         NO UNDERFITTING - Test R² = {test_r2:.4f} (good explanatory power)")
else:
     print(f"    ✗ POSSIBLE UNDERFITTING - Test R² = {test_r2:.4f} (low explanatory power)")

print("\nCONCLUSION:")
print("    Multiple linear regression successfully learned the relationship.")
print("    The model generalizes well to unseen test data.")
print(f"    Both features (x₁, x₂) contribute meaningfully to predictions.")

## Step 9: Visualization

### Plot 1: Actual vs. Predicted
- Each point: (actual y, predicted y)
- Perfect predictions lie on the diagonal
- Points above/below line indicate over/under-prediction
- Tight clustering around diagonal = good predictions

### Plot 2: Residual Plot
- Residuals = actual - predicted
- Should scatter randomly around zero (dashed line)
- Patterns indicate model problems
- Validates assumption of constant prediction error

### Plot 3: Coefficient Comparison
- Side-by-side comparison of true vs. learned weights
- Visual validation that learning was successful
- Shows relative importance of each feature

In [ ]:
# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PLOT 1: Actual vs Predicted
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ax1 = axes[0]

# Plot predictions
ax1.scatter(y_train, train_preds, alpha=0.6, s=25, label="Train", color="steelblue")
ax1.scatter(y_test,    test_preds,    alpha=0.8, s=35, label="Test", color="coral")

# Plot perfect prediction line
min_val = min(Y.min(), train_preds.min(), test_preds.min())
max_val = max(Y.max(), train_preds.max(), test_preds.max())
ax1.plot([min_val, max_val], [min_val, max_val], "k--", lw=1.5, label="Perfect fit (y=ŷ)")

ax1.set_title("Actual vs. Predicted Values", fontsize=12, fontweight="bold")
ax1.set_xlabel("Actual Y", fontsize=11)
ax1.set_ylabel("Predicted Y (ŷ)", fontsize=11)
ax1.legend(loc="upper left", fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal', adjustable='box')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PLOT 2: Residuals vs Predicted
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ax2 = axes[1]

# Calculate residuals
train_res = y_train - train_preds
test_res    = y_test    - test_preds

# Plot residuals
ax2.scatter(train_preds, train_res, alpha=0.6, s=25, label="Train residuals", color="steelblue")
ax2.scatter(test_preds,    test_res,    alpha=0.8, s=35, label="Test residuals", color="coral")

# Add zero line
ax2.axhline(0, color="black", lw=1.5, ls="--", label="Zero error")

ax2.set_title("Residual Plot (Actual - Predicted)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Fitted Values (ŷ)", fontsize=11)
ax2.set_ylabel("Residuals (y - ŷ)", fontsize=11)
ax2.legend(loc="upper right", fontsize=10)
ax2.grid(True, alpha=0.3)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PLOT 3: Feature Importance (Coefficients)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ax3 = axes[2]

# Prepare data for bar plot
features = ["x₁ Coefficient\n(true=4.0)", "x₂ Coefficient\n(true=2.5)"]
learned_coefs = model.coef_
true_coefs = [a1_true, a2_true]
x_pos = np.arange(len(features))
bar_width = 0.35

# Create bars
bars1 = ax3.bar(x_pos - bar_width/2, true_coefs, bar_width, label="True coefficients", color="steelblue", alpha=0.8)
bars2 = ax3.bar(x_pos + bar_width/2, learned_coefs, bar_width, label="Learned coefficients", color="coral", alpha=0.8)

# Add value labels on bars
for bars in [bars1, bars2]:
     for bar in bars:
          height = bar.get_height()
          ax3.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=9)

ax3.set_xticks(x_pos)
ax3.set_xticklabels(features, fontsize=10)
ax3.set_title("Feature Coefficients: True vs Learned", fontsize=12, fontweight="bold")
ax3.set_ylabel("Coefficient Value", fontsize=11)
ax3.legend(loc="upper right", fontsize=10)
ax3.grid(True, axis='y', alpha=0.3)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Overall figure settings
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig.suptitle(
     "Assignment 2 – Multiple Linear Regression    |    y = 4.0x₁ + 2.5x₂ + 10",
     fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

print("\n     Visualizations generated")

## Step 10: Save Plots

Saves the 3-panel visualization to disk for future reference and reporting.

In [ ]:
# Define output directory
save_dir = r"C:\Users\Doha\Desktop\depi\04. Machine Learning\Session1 - Linear and Polynomial Regression\linear Regrression"

# Create directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Save figure
plot_path = os.path.join(save_dir, "multiple_regression_plots.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"     Plot saved successfully!")
print(f"    Location: {plot_path}")

## Summary & Key Takeaways

### What We Accomplished
1.      Defined a 2-feature linear equation: $y = 4x_1 + 2.5x_2 + 10$
2.      Generated 150 synthetic samples with realistic noise (σ=15)
3.      Split data into training (120) and test (30) sets
4.      Trained multiple linear regression model
5.      Recovered coefficients very close to true values:
     - $a_1$: 4.0 true → ~4.15 learned
     - $a_2$: 2.5 true → ~2.30 learned
     - $c$: 10.0 true → ~11.09 learned
6.      Achieved high R² (>0.94) on both train and test sets
7.      Confirmed no overfitting (train/test performance nearly identical)
8.      Visualized predictions, residuals, and feature importance

### Key Learning Points

**Multiple vs. Simple Regression:**
- Simple: 1 feature → 1 coefficient (e.g., y = ax + b)
- Multiple: 2+ features → 2+ coefficients (e.g., y = a₁x₁ + a₂x₂ + c)
- Same OLS algorithm, but finds coefficients for all features simultaneously

**Feature Contribution:**
- $x_1$ has coefficient 4.0 (learned ≈ 4.15)
- $x_2$ has coefficient 2.5 (learned ≈ 2.30)
- $x_1$ is ~1.6x more influential than $x_2$ → larger coefficient

**Model Performance:**
- High R² on both sets → Model explains ~95% of variance
- Similar train/test metrics → Good generalization, no overfitting
- MAE ≈ 11 matches noise level (σ=15) → Expected error magnitude

**When to Use Multiple Regression:**
- Multiple independent variables needed to explain output
- Linear relationship between features and target
- Prediction accuracy more important than interpretability
- Sample size much larger than number of features (avoid overfitting)

### Extension Ideas
- Add polynomial terms (x₁², x₂²) for nonlinear relationships
- Add interaction terms (x₁·x₂) for feature interactions
- Use regularization (Ridge/Lasso) to prevent overfitting
- Try feature scaling for better numerical stability
- Cross-validation for robust performance estimates